<a href="https://colab.research.google.com/github/vlaks524/DSCC-251-Final-Project/blob/main/notebooks/ver1_RoBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

import torch
from datasets import Dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
#Loading FinancialPhraseBank file (75 Agree)
FILE_PATH = "/content/Sentences_75Agree.txt"

# read raw lines
with open(FILE_PATH, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

# parse "sentence@label"
rows = []
for line in lines:
    line = line.strip()
    if not line:
        continue

    # split from the right just in case "@" appears in text
    parts = line.rsplit("@", 1)
    if len(parts) != 2:
        continue

    text, label = parts
    text = text.strip()
    label = label.strip().lower()

    if text and label in ["positive", "neutral", "negative"]:
        rows.append((text, label))

df = pd.DataFrame(rows, columns=["text", "label"])

print("Shape:", df.shape)
print(df.head())
print("\nLabel counts:")
print(df["label"].value_counts())

In [ ]:
#Encoding the labels
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print("Classes:", list(label_encoder.classes_))
print(df.head())

In [ ]:
#Splitting into test set and pool set
X = df["text"].tolist()
y = df["label_id"].tolist()

# fixed held-out test set
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Pool size:", len(X_pool))
print("Test size:", len(X_test))

In [ ]:
#Creating initial labeled set and unlabeled pool
INITIAL_LABEL_SIZE = 60

pool_indices = np.arange(len(X_pool))

initial_indices, unlabeled_indices = train_test_split(
    pool_indices,
    train_size=INITIAL_LABEL_SIZE,
    random_state=42,
    stratify=np.array(y_pool)
)

X_labeled = [X_pool[i] for i in initial_indices]
y_labeled = [y_pool[i] for i in initial_indices]

X_unlabeled = [X_pool[i] for i in unlabeled_indices]
y_unlabeled = [y_pool[i] for i in unlabeled_indices]  # hidden during AL, but kept for oracle simulation

print("Initial labeled set:", len(X_labeled))
print("Unlabeled pool:", len(X_unlabeled))
print("Test set:", len(X_test))

In [ ]:
#RoBERTa tokenizer
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
train_df = pd.DataFrame({
    "text": X_labeled,
    "label": y_labeled
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [ ]:
#Defining Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    f1_macro = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro
    }

In [ ]:
#Building the model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)

In [ ]:
#Training setup
training_args = TrainingArguments(
    output_dir="./roberta_phrasebank_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

In [ ]:
#Training initial model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
pred_output = trainer.predict(test_dataset)
preds = np.argmax(pred_output.predictions, axis=1)

print("Macro F1:", f1_score(y_test, preds, average="macro"))
print("Accuracy:", accuracy_score(y_test, preds))
print("\nClassification report:\n")
print(classification_report(y_test, preds, target_names=label_encoder.classes_))